# DataWorks Challenge 2026: run the whole pipeline on Google Colab

This notebook runs our ML-assisted history matching and 20-year forecast, **step by step**, using files stored on
your Google Drive. You don't need to install anything on your laptop.

**How to use it:** run the cells from top to bottom. Click a cell, then press **Shift + Enter**, or press the ▶ button
on its left. Wait until a cell finishes (its ▶ stops spinning) before running the next one.

| Step | What happens | Time |
|---|---|---|
| 0 | Put the challenge files on Google Drive (done by hand, once) | 5 min |
| 1 | Connect this notebook to your Google Drive | 30 s |
| 2 | Tell the notebook where your files are | instant |
| 3 | Download our code from GitHub | 10 s |
| 4 | Install the Python packages | 30 s |
| 5 | Copy the challenge files into the project | 10 s |
| 6 | **Pipeline step 1:** data preparation & EDA | 20 s |
| 7 | **Pipeline step 2:** train the ML proxy | 1 min |
| 8 | **Pipeline step 3:** history matching | 5-10 min |
| 9 | **Pipeline step 4:** 20-year forecast | 10 s |
| 10 | **Pipeline step 5:** fill the template + Power BI tables | 10 s |
| 11 | Save everything back to Google Drive and download the deliverable | 30 s |

## Step 0: put the challenge files on Google Drive (do this once)

1. Open [drive.google.com](https://drive.google.com).
2. In **My Drive**, create a folder named **`DWC2026`**. Inside it, create a folder named **`data`**.
3. Upload these files into `DWC2026/data`. You can keep their original names:
   * `01 Introduction.xlsx`  ← required
   * `02 Field Production.xlsx`  ← required
   * `09 Template Deliverable.xlsx`  ← required
   * `03`–`08` (grid files): **optional**. The model does not need them; they are only used by the optional last
     cell.

Your Drive should then look like this:
```
My Drive
└── DWC2026
    └── data
        ├── 01 Introduction.xlsx
        ├── 02 Field Production.xlsx
        └── 09 Template Deliverable.xlsx
```

## Step 1: connect Google Drive
Run the cell below. A pop-up asks you to choose your Google account and allow access. Click **Allow**. When it
finishes, it prints `Mounted at /content/drive`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: tell the notebook where your files are
If you used the folder names from Step 0, you don't need to change anything. The cell checks that the three
required files are there.

In [ ]:
import glob, os

DRIVE_DIR = "/content/drive/MyDrive/DWC2026"     # <- your folder on Google Drive
DATA_DIR  = f"{DRIVE_DIR}/data"                  # <- the challenge Excel files are here
WORK_DIR  = "/content/dataweek"                  # where the code runs (temporary Colab disk)
REPO_URL  = "https://github.com/soulfulreflection/dataweek.git"
BRANCH    = "claude/exxonmobil-dataworks-ml-v2bxpa"

def norm(name):
    return name.lower().replace(" ", "_")

REQUIRED = {"Introduction": "introduction", "Field Production": "field_production", "Template": "template"}
files = sorted(os.path.basename(p) for p in glob.glob(f"{DATA_DIR}/*.xlsx"))
print(f"Excel files in {DATA_DIR}:")
for f in files:
    print("   ", f)
missing = [k for k, key in REQUIRED.items() if not any(key in norm(f) for f in files)]
if missing:
    raise FileNotFoundError(f"Missing file(s) containing {missing}. Upload them to {DATA_DIR} (see Step 0).")
print("\nAll required files found ✔")

## Step 3: get the code
By default the code is downloaded from our GitHub repository.

**If the repository is private** (recommended during the competition), use either option:
* **Option A (token):** create a GitHub *Personal access token* with read access to the repository. In Colab, click
  the 🔑 **Secrets** icon in the left sidebar, add a secret named `GH_TOKEN` with the token as its value, and switch
  on *Notebook access*.
* **Option B (zip):** on GitHub, open the branch, click **Code → Download ZIP**, rename the file to `dataweek.zip` and
  upload it to `My Drive/DWC2026/`. If that file exists, this cell uses it and skips GitHub.

In [ ]:
import shutil, subprocess, zipfile

CODE_ZIP = f"{DRIVE_DIR}/dataweek.zip"
shutil.rmtree(WORK_DIR, ignore_errors=True)

if os.path.exists(CODE_ZIP):
    shutil.rmtree("/content/_code", ignore_errors=True)
    with zipfile.ZipFile(CODE_ZIP) as z:
        z.extractall("/content/_code")
    inner = [p for p in glob.glob("/content/_code/*") if os.path.isdir(p)]
    shutil.move(inner[0] if len(inner) == 1 else "/content/_code", WORK_DIR)
    print("Code unzipped from Google Drive ✔")
else:
    url = REPO_URL
    try:                                   # only used when the repository is private
        from google.colab import userdata
        url = REPO_URL.replace("https://", f"https://{userdata.get('GH_TOKEN')}@")
    except Exception:
        pass
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, WORK_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr.replace(url, REPO_URL))
        raise RuntimeError("Could not download the code. Use Option A or B described above this cell.")
    print("Code downloaded from GitHub ✔")

os.chdir(WORK_DIR)
print("Working folder:", os.getcwd())
print(sorted(os.listdir()))

## Step 4: install the Python packages
Colab already has most of them, so this is quick. Warnings about pip's dependency resolver can be ignored.

In [ ]:
!pip install -q -r requirements.txt
import numpy, pandas, sklearn, scipy, openpyxl
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| scikit-learn", sklearn.__version__,
      "| scipy", scipy.__version__, "| openpyxl", openpyxl.__version__)

## Step 5: copy the challenge files into the project
The code looks for the files in `data/raw/`. The cell copies the three required ones from your Drive. It also
defines two small helpers, `show` (display figures) and `table` (display results tables), used below.

In [ ]:
import pandas as pd
from IPython.display import Image, display

os.makedirs("data/raw", exist_ok=True)
for f in glob.glob(f"{DATA_DIR}/*.xlsx"):
    if any(key in norm(os.path.basename(f)) for key in REQUIRED.values()):
        shutil.copy(f, "data/raw/")
        print("copied", os.path.basename(f))

def show(*names, width=1000):
    for n in names:
        display(Image(f"outputs/figures/{n}", width=width))

def table(name, **kw):
    return pd.read_csv(f"outputs/metrics/{name}", **kw)

## Step 6: pipeline step 1, data preparation & exploratory analysis
**What it does:** reads the observed field history and the simulated history of the 100 cases, and puts them on
the same quarterly dates (1998–2008).

**What to look for:**
* The produced GOR is constant (≈0.3632) in every case, so gas = 0.3632 × oil.
* Liquid rate is the same in all cases: the wells are on liquid-rate control.
* The porosity multiplier controls almost everything (correlation ≈0.99).
* The **water-cut collapse** plot: all cases fall on one curve, and the observed field (black dots) bends away at
  the end. That is the first sign of model error.

In [ ]:
!python -u scripts/s01_prepare_data.py
show("01_history_all_cases.png", "01_watercut_collapse.png", "01_parameter_sensitivity.png")

## Step 7: pipeline step 2, train the ML proxy
**What it does:** the proxy is a fast stand-in for the reservoir simulator. It takes the 4 uncertainty parameters
and predicts 10 years of oil and water rates. Six model types are compared on the official split: train on cases
1–70, choose on 71–85, test once on 86–100.

**What to look for:** the PCA-GP model should win, with test R² ≈ 0.997.

In [ ]:
!python -u scripts/s02_train_proxy.py
display(table("proxy_benchmark.csv").query("split == 'test'")[["model", "R2", "NRMSE_oil_%", "NRMSE_water_%"]].round(4))
show("02_proxy_benchmark.png", "02_proxy_test_cases.png")

## Step 8: pipeline step 3, history matching (the slowest step, 5–10 minutes)
**What it does:**
1. **Blind test:** each test case 86–100 pretends to be the "real field", and we check that the workflow recovers
   its known parameters.
2. **Real history match:** it screens 131,072 parameter combinations with the proxy, refines the best, samples the
   uncertainty (MCMC), and picks **10 good, different matches**. These become *Case 1 … Case 10* of the deliverable.

**What to look for:** the porosity multiplier is recovered almost perfectly, and fault transmissibility is not
(it can't be identified from production data). The field's porosity multiplier is ≈1.349.

In [ ]:
!python -u scripts/s03_history_match.py
display(table("hm_blind_test_summary.csv", index_col=0).round(1))
display(table("hm_selected_cases.csv").round(4))
show("03_history_match.png", "03_posterior_parameters.png", "03_blind_test_recovery.png")

## Step 9: pipeline step 4, the 20-year forecast (2008-04-01 → 2028-01-01)
**What it does:**
* **Backtest:** cut every simulated case off at 2006, forecast to 2008 with 5 methods, and compare with the
  simulator. The oil-cut power law wins (≈1.2% error).
* Fits the power law to the observed field (β ≈ 1.32) and forecasts the 10 cases. Gas = GOR × oil.

**What to look for:** cumulative oil ≈ 55.9 MMSTB in 2028. The dashed orange "uncorrected" forecast (≈60.9 MMSTB)
shows what you would get by trusting the model's own decline (β ≈ 0.81).

In [ ]:
!python -u scripts/s04_forecast.py
cols = ["deliverable_case", "poro_mult", "perm_mult", "aquifer_pv", "beta_forecast",
        "oil_cum_2028_mmstb", "water_cut_2028", "oil_cum_2028_uncorrected_mmstb"]
display(table("forecast_case_summary.csv")[cols].round(3))
display(table("forecast_sensitivity.csv").round(3))
show("04_forecast.png", "04_oilcut_powerlaw.png", "04_forecast_backtest.png")

## Step 10: pipeline step 5, fill the template + Power BI tables, then run the checks
**What it does:** writes the forecast into columns B–D of sheets *Case 1 … Case 10* of `09 Template Deliverable`
and checks every date and value. It also writes `dashboard_data.xlsx` for Power BI. The sanity tests should end
with `5 tests passed`.

In [ ]:
!python -u scripts/s05_export_deliverable.py
!python -u tests/test_sanity.py
display(pd.read_excel("outputs/deliverable/09_Template_Deliverable.xlsx", sheet_name="Case 1").head(8))

## Step 11: save everything to Google Drive and download the deliverable
Colab deletes its temporary disk when you close the session, so this cell copies the whole `outputs` folder to
`My Drive/DWC2026/results_<date_time>/`. It also downloads the filled template to your computer.

In [ ]:
import datetime
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
dest = f"{DRIVE_DIR}/results_{stamp}"
shutil.copytree("outputs", dest)
print("Saved to", dest)
for root, _, fs in os.walk(dest):
    print("  ", os.path.relpath(root, dest), "->", len(fs), "files")

from google.colab import files
files.download("outputs/deliverable/09_Template_Deliverable.xlsx")
files.download("outputs/powerbi/dashboard_data.xlsx")

## Optional: summarise the big grid files (03–08) for your slides
Not needed for the results. If you uploaded the grid files to `DWC2026/data`, set `RUN_GRID = True` and run the
cell. It reads them row by row, which takes a few minutes per file, and produces a small table per case and a
top-view map of case 1.

In [ ]:
RUN_GRID = False

if RUN_GRID:
    import re
    for f in glob.glob(f"{DATA_DIR}/*.xlsx"):
        if re.match(r"0[3-8]", os.path.basename(f)):
            shutil.copy(f, "data/raw/")
            print("copied", os.path.basename(f))
    !python -u tools/extract_grid_features.py --map-case 1
    display(pd.read_csv("data/processed/grid_features.csv").head())
    show("00_grid_maps_case1.png")
    shutil.copy("data/processed/grid_features.csv", f"{DRIVE_DIR}/grid_features.csv")

## Troubleshooting
| Problem | Fix |
|---|---|
| `FileNotFoundError: Missing file(s)` in Step 2 | Check the folder names in Drive (`DWC2026/data`) and that the files end in `.xlsx` |
| `Could not download the code` in Step 3 | The repository is private: use Option A (token) or Option B (zip) |
| `No such file or directory: 'scripts/...'` | Step 3 was skipped or failed. Run Steps 2–5 again |
| The session disconnected during Step 8 | Run everything again from Step 1. Colab resets its disk when it disconnects |
| You want cumulative volumes counted from 2008 instead of 1998 | Open `src/dwc/config.py` in the 📁 Files panel, set `CUMULATIVE_FROM_START = False`, save, and re-run Steps 9–11 |

**Tip:** *Runtime → Run all* runs every cell in order. Use it once you've done Step 0 and the Drive pop-up once.